In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_A11_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_A11_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_A11_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_A11_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 100 trials.
Best value (Accuracy): 0.8518
Best Hyperparameters:
  ft_lr: 0.009947815180351594
  ft_steps: 100


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

params_to_plot_FT = [
    "ft_steps", "ft_lr"
]


In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_FT)
fig_slice.show()

In [11]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
84,84,0.851771,None,2026-04-20 05:19:06.285574,0 days 00:15:59.789846
77,77,0.851771,None,2026-04-20 05:02:49.680476,0 days 00:15:57.856845
76,76,0.851771,None,2026-04-20 04:59:13.491747,0 days 00:15:56.727249
68,68,0.851771,None,2026-04-20 04:33:09.796831,0 days 00:15:59.981592
47,47,0.851771,None,2026-04-20 03:52:20.694224,0 days 00:15:54.196919
42,42,0.851771,None,2026-04-20 03:36:08.655368,0 days 00:15:54.183951
19,19,0.851667,None,2026-04-20 01:41:04.588939,0 days 00:15:50.033739
88,88,0.851667,None,2026-04-20 05:24:01.778682,0 days 00:15:56.917255
66,66,0.851667,None,2026-04-20 04:26:18.237522,0 days 00:15:51.420904
63,63,0.851667,None,2026-04-20 04:23:56.405985,0 days 00:16:03.225413


In [12]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'ft_lr': 0.009947815180351594, 'ft_steps': 100}
{'ft_lr': 0.009987043832465478, 'ft_steps': 100}
{'ft_lr': 0.009963502430051849, 'ft_steps': 100}
{'ft_lr': 0.00994717630271733, 'ft_steps': 100}
{'ft_lr': 0.009980456015236883, 'ft_steps': 100}
{'ft_lr': 0.00995230547250911, 'ft_steps': 100}
{'ft_lr': 0.009898767979518108, 'ft_steps': 100}
{'ft_lr': 0.009837520594855721, 'ft_steps': 100}
{'ft_lr': 0.009828447877558252, 'ft_steps': 100}
{'ft_lr': 0.009833686967007604, 'ft_steps': 100}
